# 01 | SGI and groundwater memory
Recompute SGI from monthly groundwater heads using deterministic 10⁻⁶ m quantization and mean ranks for ties. SGI-3/6/12 use complete trailing windows. Compare against committed CSV; recompute M₁₂ from detrended SGI-1.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import norm, spearmanr
BASE = Path.cwd() if (Path.cwd()/'data_input').exists() else Path.cwd().parent
assert (BASE/'data_input/monthly_heads_m_asl.csv').exists(), 'Open from repo root or notebooks/'
from scipy.stats import norm
heads=pd.read_csv(BASE/'data_input/monthly_heads_m_asl.csv'); stored=pd.read_csv(BASE/'data_processed/monthly_sgi.csv');meta=pd.read_csv(BASE/'data_input/wells_metadata.csv')


In [ ]:
months=pd.to_datetime(heads.Month); m=months.dt.month
parts=[]
for well in heads.columns.drop('Month'):
    h=heads[well]
    for scale in [1,3,6,12]:
        x=(h if scale==1 else h.rolling(scale,min_periods=scale).mean()).round(6)
        n=x.groupby(m).transform('count')
        r=x.groupby(m).rank(method='average')
        parts.append(pd.DataFrame({'month':heads.Month,'well':well,'scale':scale,'sgi':norm.ppf((r-.5)/n)}))
recalc=(pd.concat(parts).pivot(index=['month','well'],columns='scale',values='sgi').reset_index().rename(columns={s:f'SGI_{s}' for s in [1,3,6,12]})).sort_values(['month','well']).reset_index(drop=True)
for scale in [1,3,6,12]:
    col=f'SGI_{scale}'
    check=stored.merge(recalc,on=['month','well'],suffixes=('_stored','_recalc'),validate='one_to_one')
    np.testing.assert_allclose(check[col+'_recalc'].to_numpy(),check[col+'_stored'].to_numpy(),rtol=0,atol=1e-9,equal_nan=True)
print('SGI reproduced from monthly head inputs (all 56 wells, four scales).')


In [ ]:
wide=recalc.pivot(index='month',columns='well',values='SGI_1').sort_index()
def acf(x,maxlag=36,detrend=True):
    a=np.asarray(x,float);t=np.arange(len(a));valid=np.isfinite(a)
    if valid.sum()<maxlag+10:return np.full(maxlag+1,np.nan)
    if detrend:
        slope,intercept=np.polyfit(t[valid],a[valid],1); a=a-(slope*t+intercept)
    out=[1.]
    for lag in range(1,maxlag+1):
        ok=np.isfinite(a[:-lag])&np.isfinite(a[lag:])
        out.append(np.corrcoef(a[:-lag][ok],a[lag:][ok])[0,1] if ok.sum()>4 else np.nan)
    return np.array(out)
rows=[]
for well in wide:
    a=acf(wide[well]);cross=np.where(a[1:]<1/np.e)[0]
    rows.append([well,a[1],np.nanmean(a[1:13]),int(cross[0]+1) if len(cross) else np.nan,np.nanmean(acf(wide[well],detrend=False)[1:13])])
mem=pd.DataFrame(rows,columns=['well','detrended_ACF1','detrended_M12','detrended_first_lag_below_1e','raw_M12'])
reference=pd.read_csv(BASE/'data_processed/groundwater_memory.csv')
assert set(reference.well)==set(mem.well)
for col in mem.columns.drop('well'):
    np.testing.assert_allclose(reference.set_index('well').loc[mem.well,col].to_numpy(),mem[col].to_numpy(),rtol=0,atol=1e-9,equal_nan=True)
print(reference.query('group in ["D","M"]').groupby('group').detrended_M12.mean())


No figures are generated. Original archival SGI values differ at a small number of tied/near-tied observations; see `RECONCILIATION_PENDING.md`.